# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya - Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, with a focus on referencing all dataset entities via their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset Croissant schema is provided via this URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and explore the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata information
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"License: {dataset.metadata.license}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview

Examine the available record sets, fields, and columns, referencing them by their `@id` values as required by FAIR and Croissant standards. This overview helps you select which components to analyze further.

In [ ]:
# List all record sets in the dataset, by @id
print("Record sets found in dataset:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', '')}")

# For each record set, print its fields (by @id) and column details
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"  - field @id: {fid}")
    else:
        print("  (No fields listed)")


## 3. Data Extraction

Load data from one or more record sets into DataFrames, referencing entities by their `@id`. Identify which record sets represent tabular data suitable for further analysis.

In [ ]:
# Collect record_set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = dict()
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Display columns for the first available DataFrame
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nFirst extracted record set @id: {first_rs_id}")
    print("Columns (by field @id):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Perform basic data processing and exploration on a numeric field. All columns and operations are referenced using their `@id`.

*Examples: Remove/filter outliers, normalize a numeric field, and group by a categorical column with its `@id`.*

In [ ]:
# Choose a DataFrame by record set @id
record_set_id = None
numeric_field_id = None
group_field_id = None

# Automatically pick first DataFrame with numeric-looking columns for demonstration
for rs_id, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if numeric_cols:
        record_set_id = rs_id
        numeric_field_id = numeric_cols[0]  # Use first numeric column by field @id
        potential_cat = df.select_dtypes(include=['object']).columns.tolist()
        group_field_id = potential_cat[0] if potential_cat else None
        break

if record_set_id is not None:
    print(f"Working with record set @id: {record_set_id}")
    print(f"Numeric field selected for analysis: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping field: {group_field_id}")
    else:
        print("No obvious grouping field (categorical @id) found.")
else:
    print("No suitable numeric field found for EDA.")

# Proceed with filtering
if record_set_id and numeric_field_id:
    df = dataframes[record_set_id]
    # Example: Filter out entries with value > threshold
    threshold = df[numeric_field_id].mean()  # set threshold at mean for demonstration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field, if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        print(grouped.head())

## 5. Visualization

Visualize the distribution of the chosen numeric variable and its relationship with a grouping field, using `@id` for labeling axes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution plot
    sns.histplot(df[numeric_field_id].dropna(), ax=axs[0], kde=True, color='skyblue')
    axs[0].set_title(f'Distribution of {numeric_field_id}')
    axs[0].set_xlabel(numeric_field_id)
    axs[0].set_ylabel('Count')

    # Boxplot by group (if available)
    if group_field_id and group_field_id in df.columns:
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=axs[1])
        axs[1].set_title(f'{numeric_field_id} by {group_field_id}')
        axs[1].set_xlabel(group_field_id)
        axs[1].set_ylabel(numeric_field_id)
        axs[1].tick_params(axis='x', rotation=45)
    else:
        axs[1].axis('off')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset using the `mlcroissant` library, explored its structure via `@id` fields, extracted records for analysis, performed basic EDA and normalization, and visualized core relationships. Referencing all entities via their `@id` ensures robust, reproducible data workflows across Croissant-compliant datasets.